# Descarga de imágenes desde Google Drive (Fotos OVH)

### Antes de correr
1. Abrí la carpeta **Fotos OVH** en Google Drive
2. Copiá el ID de la URL: `https://drive.google.com/drive/folders/`**`ESTE_ES_EL_ID`**
3. Pegalo en `FOLDER_ID` abajo

In [ ]:
# ── Configuración ──────────────────────────────────────────────────────────
# Pegá la URL completa o solo el ID — ambos funcionan
FOLDER_INPUT = "https://drive.google.com/drive/folders/1_JPwdAmd7mNxxtawstxQ24Re2ZysiD_H?usp=drive_link"
OUTPUT_DIR   = "/content/fotos_ovh"

# Extraer solo el ID si pegaron la URL completa
import re
m = re.search(r'/folders/([a-zA-Z0-9_-]+)', FOLDER_INPUT)
FOLDER_ID = m.group(1) if m else FOLDER_INPUT.strip()
print(f"Folder ID: {FOLDER_ID}")

In [ ]:
# ── Autenticar con Google Drive ────────────────────────────────────────────
from google.colab import auth
auth.authenticate_user()
print("Autenticado ✓")

In [ ]:
# ── Instalar dependencias ──────────────────────────────────────────────────
%pip install -q tqdm google-api-python-client

In [ ]:
# ── Listar todas las imágenes JPEG recursivamente ──────────────────────────
from googleapiclient.discovery import build
import os

service = build('drive', 'v3')
os.makedirs(OUTPUT_DIR, exist_ok=True)

JPEG_MIMES = {'image/jpeg', 'image/jpg'}
FOLDER_MIME = 'application/vnd.google-apps.folder'

def list_jpegs(folder_id, path=""):
    results = []
    page_token = None
    while True:
        resp = service.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields="nextPageToken, files(id, name, mimeType)",
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
            pageSize=1000
        ).execute()

        for item in resp.get('files', []):
            if item['mimeType'] == FOLDER_MIME:
                sub = list_jpegs(item['id'], path=item['name'] + "_")
                results.extend(sub)
            elif item['mimeType'] in JPEG_MIMES:
                results.append((item['id'], path + item['name']))

        page_token = resp.get('nextPageToken')
        if not page_token:
            break
    return results

print(f"Listando imágenes en: {FOLDER_ID}")
all_images = list_jpegs(FOLDER_ID)
print(f"Total: {len(all_images)} imágenes JPEG encontradas")

In [ ]:
# ── Descargar todas las imágenes ───────────────────────────────────────────
from googleapiclient.http import MediaIoBaseDownload
from tqdm import tqdm
import io

errors = []
skipped = 0

for file_id, dest_name in tqdm(all_images, desc="Descargando"):
    out_path = os.path.join(OUTPUT_DIR, dest_name)

    if os.path.exists(out_path):
        skipped += 1
        continue

    try:
        request = service.files().get_media(
            fileId=file_id,
            supportsAllDrives=True
        )
        with io.FileIO(out_path, 'wb') as fh:
            dl = MediaIoBaseDownload(fh, request, chunksize=4*1024*1024)
            done = False
            while not done:
                _, done = dl.next_chunk()
    except Exception as e:
        errors.append((dest_name, str(e)))

downloaded = len(all_images) - len(errors) - skipped
print(f"\nDescargadas:  {downloaded}")
print(f"Ya existían:  {skipped}")
print(f"Errores:      {len(errors)}")
if errors:
    for name, err in errors[:10]:
        print(f"  ✗ {name}: {err}")

In [ ]:
# ── Resumen ────────────────────────────────────────────────────────────────
import glob

jpegs = glob.glob(OUTPUT_DIR + '/**/*.jpg', recursive=True) + \
        glob.glob(OUTPUT_DIR + '/**/*.jpeg', recursive=True) + \
        glob.glob(OUTPUT_DIR + '/*.jpg') + \
        glob.glob(OUTPUT_DIR + '/*.jpeg')

total_mb = sum(os.path.getsize(f) for f in jpegs) / 1e6
print(f"Imágenes en disco: {len(jpegs)}")
print(f"Tamaño total:      {total_mb:.0f} MB")
print(f"Carpeta:           {OUTPUT_DIR}")
print()
print("Próximo paso: subir estas imágenes a Roboflow para etiquetar pallets")

In [ ]:
# ── Comprimir y descargar todo como ZIP ────────────────────────────────────
import shutil, os
from google.colab import files

ZIP_PATH = "/content/fotos_ovh"

print("Comprimiendo...")
shutil.make_archive(ZIP_PATH, 'zip', OUTPUT_DIR)

zip_size = os.path.getsize(ZIP_PATH + '.zip') / 1e6
print(f"ZIP generado: {ZIP_PATH}.zip  ({zip_size:.0f} MB)")
print("Descargando...")
files.download(ZIP_PATH + '.zip')